# C6-pytorch — Practice p17 — Solution


Both subclasses register only the numbers their score needs and inherit
the thresholding template unchanged.  `PlaneGate` returns an affine
score; `BandGate` subtracts absolute line-score distance from the
allowed radius.


In [ ]:
import torch
import torch.nn as nn

torch.set_default_dtype(torch.float64)

class GateBase(nn.Module):
    """Template: forward thresholds whatever score a subclass defines."""

    def score(self, x):
        raise NotImplementedError("subclasses must define score")

    def forward(self, x):
        return (self.score(x) >= 0).to(x.dtype)


class PlaneGate(GateBase):
    def __init__(self, w, c):
        super().__init__()
        self.w = nn.Parameter(torch.as_tensor(w), requires_grad=False)
        self.c = nn.Parameter(torch.as_tensor(c), requires_grad=False)

    def score(self, x):
        return x @ self.w + self.c


class BandGate(GateBase):
    def __init__(self, w, c, r):
        super().__init__()
        self.w = nn.Parameter(torch.as_tensor(w), requires_grad=False)
        self.c = nn.Parameter(torch.as_tensor(c), requires_grad=False)
        self.r = nn.Parameter(torch.as_tensor(r), requires_grad=False)

    def score(self, x):
        return self.r - (x @ self.w + self.c).abs()


pts = torch.tensor([[0.0, 0.0], [2.0, 1.0], [1.0, 1.6],
                    [-1.0, 0.4], [3.0, 3.0]])
pg = PlaneGate(torch.tensor([1.0, -1.0]), torch.tensor([0.5]))
bg = BandGate(torch.tensor([0.0, 1.0]), torch.tensor([-1.0]), torch.tensor([0.5]))
plane_out = pg(pts)
band_out = bg(pts)

shares_forward = PlaneGate.forward is GateBase.forward
n_plane = sum(p.numel() for p in pg.parameters())
n_band = sum(p.numel() for p in bg.parameters())
both_modules = isinstance(pg, nn.Module) and isinstance(bg, nn.Module)

plane_out, band_out, shares_forward, n_plane, n_band, both_modules


### Answer check


In [ ]:
assert torch.equal(plane_out, torch.tensor([1.0, 1.0, 0.0, 0.0, 1.0]))
assert torch.equal(band_out, torch.tensor([0.0, 1.0, 0.0, 0.0, 0.0]))
assert shares_forward is True
assert n_plane == 3 and n_band == 4
assert both_modules is True
